# Model Evaluation

In [34]:
import lightgbm as lgb
import polars as pl
import json
import numpy as np
from sklearn.metrics import root_mean_squared_error, r2_score, mean_absolute_error
from constants import PROCESSED_DATA_FOLDER, MODEL_FOLDER, DAY_CATEGORIES, DATA_FILE
from helpers import clean_df, create_travel_time_column

In [2]:
# Cast "Route" as ENUM type to ensure identical categorical variable encoding for train and test set
with open(PROCESSED_DATA_FOLDER / "target_routes.json", "r", encoding="utf-8") as f:
    routes: list[str] = json.load(f)
with open(PROCESSED_DATA_FOLDER / "target_stops.json", "r", encoding="utf-8") as f:
    target_stops: dict[str, list[int]] = json.load(f)
with open(PROCESSED_DATA_FOLDER / "stops_dict.json", "r", encoding="utf-8") as f:
    stops_dict: dict[str, str] = json.load(f)
with open(PROCESSED_DATA_FOLDER / "target_routes.json", "r", encoding="utf-8") as f:
    target_routes: list[str] = json.load(f)
target_encoding = pl.read_csv(PROCESSED_DATA_FOLDER / "mean_travel_time_encoding.csv")
Routes_enum = pl.Enum(routes)
Days_enum = pl.Enum(DAY_CATEGORIES)

Compare with single route single segment prediction

In [ ]:
naive_model = ...
encoding_model = lgb.Booster(
    model_file=MODEL_FOLDER / "target_encoding_model/best_lgbm_model.txt"
)

In [ ]:
small_model = lgb.Booster(
    model_file=MODEL_FOLDER / "weight_model/weight_model_small.txt"
)
big_model = lgb.Booster(model_file=MODEL_FOLDER / "weight_model/weight_model_big.txt")

In [4]:
def evaluate_route_naive(route: str) -> None:

    def _prepare_evaluation_data(
        route: str, depart_id: int, arrival_id: int
    ) -> tuple[np.ndarray, np.ndarray]:
        test = (
            pl.read_parquet(PROCESSED_DATA_FOLDER / "test.parquet")
            .filter(
                pl.col("Route") == route,
                pl.col("DepartureStop") == depart_id,
                pl.col("ArrivalStop") == arrival_id,
            )
            .with_columns(
                pl.col("Route").cast(Routes_enum).to_physical(),
                pl.col("DayOfWeek").cast(Days_enum).to_physical(),
            )
            .cast({"MinutesFromMidnight": pl.Float16})
        )
        return (
            test.drop("TravelDuration_min").to_numpy(),
            test["TravelDuration_min"].to_numpy(),
        )

    def _pairwise(lst):
        return list(zip(lst, lst[1:]))

    def _print_tables(route: str, depart_id: int, arrival_id: int) -> None:
        x, y = _prepare_evaluation_data(route, depart_id, arrival_id)
        output = pl.DataFrame({"prediction": naive_model.predict(x), "label": y})
        average_time = np.mean(y)
        loose_criterion = average_time * 0.1
        strict_criterion = average_time * 0.05
        residuals = (output["prediction"] - output["label"]).abs()

        print(
            f"Between {stops_dict[str(depart_id)]} and {stops_dict[str(arrival_id)]}, average travel time is {average_time:.1f} minutes"
        )
        display(
            output.select(
                (
                    (
                        abs(pl.col("prediction") - pl.col("label")) < loose_criterion
                    ).mean()
                    * 100
                )
                .round(2)
                .alias("loose_criterion (%)"),
                (
                    (
                        abs(pl.col("prediction") - pl.col("label")) < strict_criterion
                    ).mean()
                    * 100
                )
                .round(2)
                .alias("strict_criterion (%)"),
                pl.lit(residuals.quantile(0.9)).round(2).alias("p90_coverage_error"),
                pl.lit(r2_score(output["label"], output["prediction"]))
                .round(2)
                .alias("r2"),
                pl.lit(root_mean_squared_error(output["label"], output["prediction"]))
                .round(2)
                .alias("rmse"),
                pl.lit(mean_absolute_error(output["label"], output["prediction"]))
                .round(2)
                .alias("mae"),
            )
        )

    for pair in _pairwise(target_stops[route]):
        _print_tables(route, *pair)

In [ ]:
def evaluate_route_target_encoding(route: str) -> None:

    def _prepare_evaluation_data(
        route: str, depart_id: int, arrival_id: int
    ) -> tuple[np.ndarray, np.ndarray]:
        test = (
            pl.read_parquet(PROCESSED_DATA_FOLDER / "test.parquet")
            .filter(
                pl.col("Route") == route,
                pl.col("DepartureStop") == depart_id,
                pl.col("ArrivalStop") == arrival_id,
            )
            .join(
                target_encoding,
                on=["Route", "DepartureStop", "ArrivalStop"],
                how="left",
            )
            .with_columns(
                pl.col("Route").cast(Routes_enum).to_physical(),
                pl.col("DayOfWeek").cast(Days_enum).to_physical(),
            )
            .cast({"MinutesFromMidnight": pl.Float16})
            .select(
                [
                    "Route",
                    "TravelDuration",
                    "MinutesFromMidnight",
                    "DayOfWeek",
                    "TravelDuration_min",
                ]
            )
        )
        return (
            test.drop("TravelDuration_min").to_numpy(),
            test["TravelDuration_min"].to_numpy(),
        )

    def _pairwise(lst):
        return list(zip(lst, lst[1:]))

    def _print_tables(route: str, depart_id: int, arrival_id: int) -> None:
        x, y = _prepare_evaluation_data(route, depart_id, arrival_id)
        output = pl.DataFrame({"prediction": encoding_model.predict(x), "label": y})
        average_time = np.mean(y)
        loose_criterion = average_time * 0.1
        strict_criterion = average_time * 0.05
        residuals = (output["prediction"] - output["label"]).abs()

        print(
            f"Between {stops_dict[str(depart_id)]} and {stops_dict[str(arrival_id)]}, average travel time is {average_time:.1f} minutes"
        )
        display(
            output.select(
                (
                    (
                        abs(pl.col("prediction") - pl.col("label")) < loose_criterion
                    ).mean()
                    * 100
                )
                .round(2)
                .alias("loose_criterion (%)"),
                (
                    (
                        abs(pl.col("prediction") - pl.col("label")) < strict_criterion
                    ).mean()
                    * 100
                )
                .round(2)
                .alias("strict_criterion (%)"),
                pl.lit(residuals.quantile(0.9)).round(2).alias("p90_coverage_error"),
                pl.lit(r2_score(output["label"], output["prediction"]))
                .round(2)
                .alias("r2"),
                pl.lit(root_mean_squared_error(output["label"], output["prediction"]))
                .round(2)
                .alias("rmse"),
                pl.lit(mean_absolute_error(output["label"], output["prediction"]))
                .round(2)
                .alias("mae"),
            )
        )

    for pair in _pairwise(target_stops[route]):
        _print_tables(route, *pair)

In [ ]:
def evaluate_route_weight(route: str, model: lgb.Booster) -> None:

    def _prepare_evaluation_data(
        route: str, depart_id: int, arrival_id: int
    ) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        weekday_test = (
            pl.read_parquet(PROCESSED_DATA_FOLDER / "test.parquet")
            .filter(
                pl.col("Route") == route,
                pl.col("DepartureStop") == depart_id,
                pl.col("ArrivalStop") == arrival_id,
                ~pl.col("DayOfWeek").is_in(["Sun", "Sat"]),
            )
            .join(
                target_encoding,
                on=["Route", "DepartureStop", "ArrivalStop"],
                how="left",
            )
            .with_columns(
                pl.col("Route").cast(Routes_enum).to_physical(),
                pl.col("DayOfWeek").cast(Days_enum).to_physical(),
            )
            .cast({"MinutesFromMidnight": pl.Float16})
            .select(
                [
                    "Route",
                    "TravelDuration",
                    "MinutesFromMidnight",
                    "DayOfWeek",
                    "TravelDuration_min",
                ]
            )
        )
        weekend_test = (
            pl.read_parquet(PROCESSED_DATA_FOLDER / "test.parquet")
            .filter(
                pl.col("Route") == route,
                pl.col("DepartureStop") == depart_id,
                pl.col("ArrivalStop") == arrival_id,
                pl.col("DayOfWeek").is_in(["Sun", "Sat"]),
            )
            .join(
                target_encoding,
                on=["Route", "DepartureStop", "ArrivalStop"],
                how="left",
            )
            .with_columns(
                pl.col("Route").cast(Routes_enum).to_physical(),
                pl.col("DayOfWeek").cast(Days_enum).to_physical(),
            )
            .cast({"MinutesFromMidnight": pl.Float16})
            .select(
                [
                    "Route",
                    "TravelDuration",
                    "MinutesFromMidnight",
                    "DayOfWeek",
                    "TravelDuration_min",
                ]
            )
        )
        return (
            weekday_test.drop("TravelDuration_min").to_numpy(),
            weekday_test["TravelDuration_min"].to_numpy(),
            weekend_test.drop("TravelDuration_min").to_numpy(),
            weekend_test["TravelDuration_min"].to_numpy(),
        )

    def _pairwise(lst):
        return list(zip(lst, lst[1:]))

    def _print_tables(route: str, depart_id: int, arrival_id: int) -> None:
        weekday_x, weekday_y, weekend_x, weekend_y = _prepare_evaluation_data(
            route, depart_id, arrival_id
        )
        weekday_output = pl.DataFrame(
            {
                "prediction": model.predict(weekday_x),
                "label": weekday_y,
            }
        )
        weekend_output = pl.DataFrame(
            {
                "prediction": model.predict(weekend_x),
                "label": weekend_y,
            }
        )

        average_time = np.mean(np.append(weekday_y, weekend_y))
        loose_criterion = average_time * 0.1
        strict_criterion = average_time * 0.05

        print(
            f"Between {stops_dict[str(depart_id)]} and {stops_dict[str(arrival_id)]}, average travel time is {average_time:.1f} minutes"
        )

        metrics_rows = []
        for split, df in (("weekday", weekday_output), ("weekend", weekend_output)):
            residuals = (df["prediction"] - df["label"]).abs()

            metrics_rows.append(
                df.select(
                    pl.lit(split).alias("split"),
                    (
                        (
                            abs(pl.col("prediction") - pl.col("label"))
                            < loose_criterion
                        ).mean()
                        * 100
                    )
                    .round(2)
                    .alias("loose_criterion (%)"),
                    (
                        (
                            abs(pl.col("prediction") - pl.col("label"))
                            < strict_criterion
                        ).mean()
                        * 100
                    )
                    .round(2)
                    .alias("strict_criterion (%)"),
                    pl.lit(residuals.quantile(0.9))
                    .round(2)
                    .alias("p90_coverage_error"),
                    pl.lit(r2_score(df["label"], df["prediction"]))
                    .round(2)
                    .alias("r2"),
                    pl.lit(root_mean_squared_error(df["label"], df["prediction"]))
                    .round(2)
                    .alias("rmse"),
                    pl.lit(mean_absolute_error(df["label"], df["prediction"]))
                    .round(2)
                    .alias("mae"),
                ).head(1)
            )

        display(pl.concat(metrics_rows))

    print("===== Start of Report =====")
    for pair in _pairwise(target_stops[route]):
        _print_tables(route, *pair)
    print("===== End of Report =====")

In [13]:
r = "172801"
models = [encoding_model, big_model]
for m in models:
    evaluate_route_weight(r, m)

===== Start of Report =====
Between 仁愛復興路口(福華飯店) and 捷運大坪林站, average travel time is 25.0 minutes


split,loose_criterion (%),strict_criterion (%),p90_coverage_error,r2,rmse,mae
str,f64,f64,f64,f64,f64,f64
"""weekday""",62.5,39.0,4.48,0.21,3.06,2.25
"""weekend""",54.55,35.66,4.63,0.1,3.02,2.42


Between 捷運大坪林站 and 龍潭運動公園, average travel time is 50.8 minutes


split,loose_criterion (%),strict_criterion (%),p90_coverage_error,r2,rmse,mae
str,f64,f64,f64,f64,f64,f64
"""weekday""",64.95,37.99,8.44,0.49,6.07,4.51
"""weekend""",60.14,32.17,11.79,0.21,10.26,6.28


Between 龍潭運動公園 and 新竹轉運站, average travel time is 50.3 minutes


...

Between 捷運大坪林站 and 龍潭運動公園, average travel time is 50.8 minutes


split,loose_criterion (%),strict_criterion (%),p90_coverage_error,r2,rmse,mae
str,f64,f64,f64,f64,f64,f64
"""weekday""",65.44,38.97,8.66,0.51,5.93,4.39
"""weekend""",57.34,39.86,10.93,0.28,9.82,5.87


Between 龍潭運動公園 and 新竹轉運站, average travel time is 50.3 minutes


split,loose_criterion (%),strict_criterion (%),p90_coverage_error,r2,rmse,mae
str,f64,f64,f64,f64,f64,f64
"""weekday""",72.45,41.58,8.69,0.7,6.34,4.24
"""weekend""",78.62,40.69,8.37,0.43,6.32,4.14


===== End of Report =====
